<a href="https://colab.research.google.com/github/mujasss/Tugas-4-Sistem-temu-kembali/blob/main/240210502003_Muja_Adila_Setiawan_Preprocessing_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Nama  : Muja Adila Setiawan

NIM   : 240210502003

Tugas 4 No. 1 - Preprocessing Teks (Tokenisasi, Stopwords Removal, Stemming)
Library yang digunakan: re, Sastrawi, dan library standar Python.


In [29]:
# Install library Sastrawi untuk stopwords removal & stemming Bahasa Indonesia
!pip install Sastrawi -q

In [24]:
import re  # untuk cleaning teks dengan regular expression
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi objek stopword remover Sastrawi (dipakai di fungsi remove_stopwords)
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

# Inisialisasi objek stemmer Sastrawi (dipakai di fungsi stemming)
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

In [17]:
# -----------------------------------------------------------------------------
# Dataset: minimal 5 dokumen berbahasa Indonesia.
# Dibuat sendiri (bukan dari sumber online) dengan tema teknologi, komputer,
# dan pendidikan, meniru gaya berita/pengumuman kampus sesuai studi kasus soal.
# -----------------------------------------------------------------------------
dokumen = [
    # Dokumen 1 - tema pendidikan/administrasi kampus
    """Fakultas Teknik UNM mengumumkan bahwa pendaftaran Ujian Akhir Semester (UAS)
    Tahun Akademik 2025/2026 akan dibuka mulai tanggal 1 Desember 2025. Seluruh
    mahasiswa diwajibkan untuk melakukan pendaftaran secara online melalui SIAKAD
    sebelum batas waktu yang telah ditentukan!""",

    # Dokumen 2 - tema teknologi (kecerdasan buatan)
    """Perkembangan teknologi kecerdasan buatan atau Artificial Intelligence (AI)
    saat ini berkembang sangat pesat di berbagai bidang, termasuk pendidikan,
    kesehatan, dan industri. Banyak perusahaan teknologi besar mulai
    mengembangkan sistem AI untuk membantu pekerjaan manusia sehari-hari.""",

    # Dokumen 3 - tema pendidikan & teknologi (pelatihan pemrograman)
    """Program Studi Teknik Komputer membuka kesempatan bagi mahasiswa baru untuk
    mengikuti pelatihan pemrograman Python dan Machine Learning. Pelatihan ini
    akan dilaksanakan secara gratis selama 3 bulan bagi 50 peserta pertama yang
    mendaftar.""",

    # Dokumen 4 - tema komputer (sistem temu kembali informasi)
    """Sistem temu kembali informasi (Information Retrieval) merupakan salah satu
    cabang ilmu komputer yang mempelajari bagaimana cara menemukan kembali
    dokumen-dokumen yang relevan terhadap suatu kebutuhan informasi pengguna
    dari sebuah koleksi dokumen yang sangat besar.""",

    # Dokumen 5 - tema pengumuman kampus (fasilitas komputer)
    """Laboratorium Komputer Fakultas Teknik akan melakukan pemeliharaan rutin
    pada hari Sabtu, 15 November 2025 mulai pukul 08.00 WIB hingga selesai.
    Mahasiswa diharapkan tidak menggunakan fasilitas laboratorium selama
    kegiatan pemeliharaan berlangsung.""",
]

print("Jumlah dokumen:", len(dokumen))

Jumlah dokumen: 5


In [26]:
def case_folding(text):
    """
    Tahap 1: Case Folding
    Mengubah semua huruf pada teks menjadi huruf kecil (lowercase),
    supaya kata yang sama dengan kapitalisasi berbeda dianggap identik.
    Contoh: 'Sistem' dan 'sistem' -> 'sistem'
    """
    return text.lower()


def cleaning(text):
    """
    Tahap 2: Cleaning
    Menghapus angka, tanda baca, dan karakter khusus dari teks,
    hanya menyisakan huruf alfabet dan spasi.
    """
    text = re.sub(r"[^a-z\s]", " ", text)     # hapus semua karakter selain huruf a-z dan spasi
    text = re.sub(r"\s+", " ", text).strip()  # rapikan spasi ganda/berlebih menjadi satu spasi
    return text


def tokenize(text):
    """
    Tahap 3: Tokenisasi
    Memecah teks (string) menjadi list token/kata berdasarkan spasi.
    """
    return text.split()


def remove_stopwords(tokens):
    """
    Tahap 4: Stopwords Removal (menggunakan Sastrawi)
    Menghapus kata-kata umum yang tidak bermakna penting (stopwords),
    seperti 'yang', 'dan', 'untuk', dsb.
    """
    cleaned_sentence = stopword_remover.remove(" ".join(tokens))
    return cleaned_sentence.split()


def stemming(tokens):
    """
    Tahap 5: Stemming (menggunakan Sastrawi)
    Mengubah setiap kata berimbuhan menjadi kata dasarnya.
    Contoh: 'mengumumkan' -> 'umum', 'pendaftaran' -> 'daftar'
    """
    return [stemmer.stem(token) for token in tokens if token != ""]


def preprocess_text(text):
    """
    Fungsi utama pipeline preprocessing teks.
    Menjalankan seluruh tahapan secara berurutan sesuai instruksi soal:
    case folding -> cleaning -> tokenisasi -> stopwords removal -> stemming
    Parameter:
        text (str): teks/dokumen mentah yang akan diproses.
    Return:
        list: daftar token hasil akhir setelah seluruh tahap preprocessing.
    """
    text = case_folding(text)       # tahap 1
    text = cleaning(text)           # tahap 2
    tokens = tokenize(text)         # tahap 3
    tokens = remove_stopwords(tokens)  # tahap 4
    tokens = stemming(tokens)       # tahap 5
    tokens = [t for t in tokens if t != ""]  # buang token kosong (jika ada)
    return tokens

In [27]:
# -----------------------------------------------------------------------------
# Poin 2: Terapkan preprocess_text() pada ke-5 dokumen.
# Sekalian dihitung jumlah token sebelum & sesudah untuk kebutuhan poin 4.
# -----------------------------------------------------------------------------
hasil_preprocessing = []   # menyimpan hasil akhir (list token) tiap dokumen
token_sebelum_list = []    # menyimpan token "sebelum" (split spasi dari teks mentah)
statistik = []             # menyimpan data jumlah token & persentase reduksi

for i, doc in enumerate(dokumen, start=1):
    # "Sebelum preprocessing" = tokenisasi naif (split spasi) dari teks ASLI,
    # merepresentasikan kondisi data sebelum masuk pipeline preprocess_text() sama sekali
    token_sebelum = doc.split()
    jumlah_sebelum = len(token_sebelum)

    # "Sesudah preprocessing" = hasil akhir dari fungsi preprocess_text()
    token_sesudah = preprocess_text(doc)
    jumlah_sesudah = len(token_sesudah)

    # Hitung persentase pengurangan jumlah token
    persen_reduksi = ((jumlah_sebelum - jumlah_sesudah) / jumlah_sebelum) * 100 if jumlah_sebelum else 0

    # Simpan semua hasil ke list masing-masing untuk dipakai di sel berikutnya
    token_sebelum_list.append(token_sebelum)
    hasil_preprocessing.append(token_sesudah)
    statistik.append({
        "dokumen": "Dokumen " + str(i),
        "jumlah_token_sebelum": jumlah_sebelum,
        "jumlah_token_sesudah": jumlah_sesudah,
        "persentase_pengurangan": round(persen_reduksi, 2),
    })

print("Preprocessing selesai untuk", len(dokumen), "dokumen.")

Preprocessing selesai untuk 5 dokumen.


In [28]:
# -----------------------------------------------------------------------------
# Poin 3: Tampilkan perbandingan sebelum-sesudah preprocessing untuk SETIAP
# dokumen. Dokumen 1 & 2 ditampilkan LENGKAP (semua tahapan), Dokumen 3-5
# ditampilkan versi ringkas (sebelum vs sesudah saja) agar output tetap rapi.
# -----------------------------------------------------------------------------
print("=" * 80)
print("PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING")
print("=" * 80)

# --- Dokumen 1 & 2: ditampilkan lengkap semua tahap pipeline ---
for i in range(2):
    print("\n--- Dokumen " + str(i + 1) + " ---")
    print("Teks Asli (SEBELUM preprocessing):")
    print(dokumen[i].strip())

    print("\nToken sebelum preprocessing (split spasi saja):")
    print(token_sebelum_list[i])

    print("\nSetelah Case Folding + Cleaning + Tokenisasi:")
    print(tokenize(cleaning(case_folding(dokumen[i]))))

    print("\nSetelah Stopwords Removal + Stemming (SESUDAH preprocessing / hasil akhir):")
    print(hasil_preprocessing[i])
    print("-" * 80)

# --- Dokumen 3-5: perbandingan ringkas (sebelum vs sesudah saja) ---
print("\n--- Perbandingan Sebelum vs Sesudah untuk Dokumen 3-5 (versi ringkas) ---")
for i in range(2, 5):
    print("\nDokumen " + str(i + 1))
    print("  Sebelum :", token_sebelum_list[i])
    print("  Sesudah :", hasil_preprocessing[i])

PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING

--- Dokumen 1 ---
Teks Asli (SEBELUM preprocessing):
Fakultas Teknik UNM mengumumkan bahwa pendaftaran Ujian Akhir Semester (UAS)
    Tahun Akademik 2025/2026 akan dibuka mulai tanggal 1 Desember 2025. Seluruh
    mahasiswa diwajibkan untuk melakukan pendaftaran secara online melalui SIAKAD
    sebelum batas waktu yang telah ditentukan!

Token sebelum preprocessing (split spasi saja):
['Fakultas', 'Teknik', 'UNM', 'mengumumkan', 'bahwa', 'pendaftaran', 'Ujian', 'Akhir', 'Semester', '(UAS)', 'Tahun', 'Akademik', '2025/2026', 'akan', 'dibuka', 'mulai', 'tanggal', '1', 'Desember', '2025.', 'Seluruh', 'mahasiswa', 'diwajibkan', 'untuk', 'melakukan', 'pendaftaran', 'secara', 'online', 'melalui', 'SIAKAD', 'sebelum', 'batas', 'waktu', 'yang', 'telah', 'ditentukan!']

Setelah Case Folding + Cleaning + Tokenisasi:
['fakultas', 'teknik', 'unm', 'mengumumkan', 'bahwa', 'pendaftaran', 'ujian', 'akhir', 'semester', 'uas', 'tahun', 'akademik', 'akan',

In [21]:
# -----------------------------------------------------------------------------
# Poin 4: Hitung dan tampilkan jumlah token sebelum preprocessing, jumlah token
# setelah preprocessing, dan persentase pengurangannya (dalam bentuk tabel).
# Dibuat manual (tanpa pandas) karena soal No.1 hanya mengizinkan library
# re, Sastrawi, dan library standar Python.
# -----------------------------------------------------------------------------
print("=" * 80)
print("STATISTIK JUMLAH TOKEN")
print("=" * 80)

header = "{:<12}{:>16}{:>16}{:>14}".format("Dokumen", "Token Sebelum", "Token Sesudah", "Reduksi (%)")
print(header)
print("-" * len(header))

for s in statistik:
    print("{:<12}{:>16}{:>16}{:>14}".format(
        s["dokumen"], s["jumlah_token_sebelum"], s["jumlah_token_sesudah"], s["persentase_pengurangan"]
    ))

# Rata-rata persentase pengurangan token dari seluruh dokumen
rata_rata_reduksi = sum(s["persentase_pengurangan"] for s in statistik) / len(statistik)
print("\nRata-rata persentase pengurangan token: {:.2f}%".format(rata_rata_reduksi))

STATISTIK JUMLAH TOKEN
Dokumen        Token Sebelum   Token Sesudah   Reduksi (%)
----------------------------------------------------------
Dokumen 1                 36              27          25.0
Dokumen 2                 34              30         11.76
Dokumen 3                 32              23         28.12
Dokumen 4                 33              27         18.18
Dokumen 5                 30              24          20.0

Rata-rata persentase pengurangan token: 20.61%


In [22]:
# -----------------------------------------------------------------------------
# Poin 5: Analisis singkat (maksimal 1 paragraf) mengenai dampak preprocessing
# terhadap kualitas data untuk keperluan sistem IR (Information Retrieval).
# -----------------------------------------------------------------------------
analisis = """
ANALISIS SINGKAT:
Preprocessing (case folding, cleaning, tokenisasi, stopwords removal, dan stemming)
terbukti secara signifikan mengurangi jumlah token pada setiap dokumen, terutama karena
banyaknya stopwords (seperti yang, akan, untuk, dan) yang tidak membawa informasi
penting bagi topik dokumen. Proses stemming juga menyatukan kata-kata berimbuhan menjadi
kata dasar (misalnya mengumumkan menjadi umum, dilaksanakan menjadi laksana),
sehingga variasi bentuk kata yang sebenarnya merujuk pada makna yang sama dapat digabung
menjadi satu representasi term. Dampaknya bagi sistem temu kembali informasi (IR) sangat
positif: ukuran vocabulary menjadi lebih kecil dan efisien, pencocokan (matching) antara
query dan dokumen menjadi lebih akurat karena tidak terpengaruh oleh bentuk kata yang
berbeda-beda, serta beban komputasi untuk pembobotan (misalnya TF-IDF) pada tahap
selanjutnya menjadi lebih ringan tanpa kehilangan makna inti dari dokumen.
"""
print(analisis)


ANALISIS SINGKAT:
Preprocessing (case folding, cleaning, tokenisasi, stopwords removal, dan stemming)
terbukti secara signifikan mengurangi jumlah token pada setiap dokumen, terutama karena
banyaknya stopwords (seperti yang, akan, untuk, dan) yang tidak membawa informasi
penting bagi topik dokumen. Proses stemming juga menyatukan kata-kata berimbuhan menjadi
kata dasar (misalnya mengumumkan menjadi umum, dilaksanakan menjadi laksana),
sehingga variasi bentuk kata yang sebenarnya merujuk pada makna yang sama dapat digabung
menjadi satu representasi term. Dampaknya bagi sistem temu kembali informasi (IR) sangat
positif: ukuran vocabulary menjadi lebih kecil dan efisien, pencocokan (matching) antara
query dan dokumen menjadi lebih akurat karena tidak terpengaruh oleh bentuk kata yang
berbeda-beda, serta beban komputasi untuk pembobotan (misalnya TF-IDF) pada tahap
selanjutnya menjadi lebih ringan tanpa kehilangan makna inti dari dokumen.

